# 03 - Dimensions Exploration

This notebook explores categorical business dimensions in the gold layer.

Focus areas:
- Customer geography and demographics
- Product categories and product lines
- Relationship between customers, products and sales
- Distribution and contribution of key business dimensions

In [0]:
%sql
/*
Customer Dimension Exploration

Purpose:
    Explore the main categorical attributes in the customer dimension.
    This helps understand customer distribution by country, gender,
    and marital status before connecting customers to sales activity.
*/

SELECT
    country,
    gender,
    marital_status,
    COUNT(customer_id) AS customer_count
FROM datawarehouseanalytics_gold.dim_customers
GROUP BY country, gender, marital_status
ORDER BY customer_count DESC;

In [0]:
%sql
/*

Customer Distribution by Country
Purpose:
    Measure customer concentration by country and calculate each country's
    percentage contribution to the total customer base.

*/

SELECT
    country,
    COUNT(customer_id) AS customer_count,
    ROUND(
        COUNT(customer_id) * 100.0 / SUM(COUNT(customer_id)) OVER (),
        3
    ) AS customer_percentage
FROM datawarehouseanalytics_gold.dim_customers
GROUP BY country
ORDER BY customer_count DESC;

In [0]:
%sql
/*

Product Dimension Exploration
Purpose:
    Explore product hierarchy across category, subcategory, and product line.
    This helps identify how products are organized for sales analysis.

*/

SELECT DISTINCT
    COALESCE(category, 'Unknown') AS category,
    COALESCE(subcategory, 'Unknown') AS subcategory,
    product_line,
    COUNT(product_id) AS product_count,
    ROUND(AVG(cost), 2) AS avg_cost,
    MIN(cost) AS min_cost,
    MAX(cost) AS max_cost
FROM datawarehouseanalytics_gold.dim_products
GROUP BY category, subcategory, product_line
ORDER BY category, product_count DESC;

In [0]:
%sql
/*
Missing Product Classification Check

Purpose:
    Identify products with missing category or subcategory values.
    This helps detect classification gaps before using product dimensions
    in sales analysis.
*/

SELECT
    product_key,
    product_id,
    product_name,
    category,
    subcategory,
    product_line,
    cost
FROM datawarehouseanalytics_gold.dim_products
WHERE category IS NULL
   OR subcategory IS NULL;

In [0]:
%sql
/*
Product Category Distribution

Purpose:
    Calculate how product records are distributed across categories.
    This helps understand the structure of the product catalog.
*/

SELECT
    COALESCE(category, 'Unknown') AS category,
    COUNT(product_id) AS product_count,
    ROUND(
        COUNT(product_id) * 100.0 / SUM(COUNT(product_id)) OVER (),
        3
    ) AS product_percentage
FROM datawarehouseanalytics_gold.dim_products
GROUP BY COALESCE(category, 'Unknown')
ORDER BY product_count DESC;